In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn missingno -q

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno   # 결측 패턴 시각화 전용 라이브러리 (Part 3에서 사용)

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (운영체제별 분기)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.5.0
pandas: 3.0.3


In [2]:
# 5종 세트 ② — 열별 결측 비율(%) — '많고 적음'을 직관적으로
def missing_summary(df):
    s = df.isnull().sum()
    p = (df.isnull().mean() * 100).round(2)
    out = pd.DataFrame({"missing": s, "missing_pct(%)": p})
    return out[out["missing"] > 0].sort_values("missing", ascending=False)


In [3]:
# IQR을 재사용 가능한 함수로 만들기 — 분석가의 일상 도구
def detect_outliers_iqr(series, k=1.5):
    '''IQR 방법으로 이상치 마스크와 경계를 반환합니다.

    Parameters
    ----------
    series : pd.Series  — 수치형 시리즈
    k      : float      — 경계 폭의 IQR 배수 (기본 1.5, 더 엄격하게 보려면 3)

    Returns
    -------
    mask   : pd.Series(bool)  — 이상치 위치 (True)
    bounds : (lower, upper)   — 경계값
    '''
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    mask = (series < lower) | (series > upper)
    return mask, (lower, upper)


In [4]:
# 새 데이터셋 — '옷장패션' 주문 (가상)
np.random.seed(11)
n = 1500

partner = pd.DataFrame({
    "order_id": [f"K{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(33, 8, n).round().astype(int),
    "category": np.random.choice(["상의", "하의", "신발", "액세서리"], n, p=[0.35, 0.3, 0.2, 0.15]),
    "channel": np.random.choice(["web", "app"], n, p=[0.4, 0.6]),
    "price": np.random.choice([15900, 29900, 49900, 79900, 129900], n),
    "quantity": np.random.choice([1, 1, 1, 2, 2, 3], n),
})
partner["amount"] = partner["price"] * partner["quantity"]
partner["return_amount"] = np.where(
    np.random.rand(n) < 0.07, partner["amount"] * np.random.uniform(0.5, 1.0, n), 0
).round(0)

# 오염 심기
# (a) 나이 이상치 — 입력 실수(0, 999)
partner.loc[partner.sample(3, random_state=1).index, "customer_age"] = 999
partner.loc[partner.sample(2, random_state=2).index, "customer_age"] = 0

# (b) amount 결측 — app 채널에 더 자주 (MAR 시그널)
app = partner["channel"] == "app"
partner.loc[partner[app].sample(frac=0.05, random_state=3).index, "amount"] = np.nan
partner.loc[partner[~app].sample(frac=0.01, random_state=4).index, "amount"] = np.nan

# (c) return_amount 결측은 그대로 (0=환불없음)이라 결측 아님. 단, '관찰 안 됨'을 의도적으로 표현하기 위해
#     price 결측 5건 추가(접속 시점 가격이 누락된 사례)
partner.loc[partner.sample(5, random_state=5).index, "price"] = np.nan

# (d) quantity 이상치(단일 소비자 200개)
partner.loc[partner.sample(1, random_state=6).index, "quantity"] = 200

# (e) amount 극단값(50,000,000짜리 한 건 — '도매 의심')
partner.loc[partner.sample(1, random_state=7).index, "amount"] = 50_000_000

print("옷장패션 데이터 준비 완료:", partner.shape)
partner.head()

옷장패션 데이터 준비 완료: (1500, 8)


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
0,K00001,47,신발,app,29900.0,2,59800.0,45445.0
1,K00002,31,상의,app,129900.0,3,389700.0,0.0
2,K00003,29,상의,web,49900.0,2,99800.0,0.0
3,K00004,12,상의,web,49900.0,3,149700.0,0.0
4,K00005,33,하의,app,129900.0,1,129900.0,0.0


In [5]:
# 시나리오 1 — 진단
print("shape:", partner.shape)
partner.info()
display(partner.describe())

shape: (1500, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       1500 non-null   str    
 1   customer_age   1500 non-null   int64  
 2   category       1500 non-null   str    
 3   channel        1500 non-null   str    
 4   price          1495 non-null   float64
 5   quantity       1500 non-null   int64  
 6   amount         1449 non-null   float64
 7   return_amount  1500 non-null   float64
dtypes: float64(3), int64(2), str(3)
memory usage: 93.9 KB


,customer_age,price,quantity,amount,return_amount
count,1500.000000,1495.000000,1500.000000,1.449000e+03,1500.000000
mean,34.903333,60960.869565,1.789333,1.350475e+05,6381.010000
std,43.936525,40275.103681,5.173406,1.313695e+06,28315.037316
min,0.000000,15900.000000,1.000000,1.590000e+04,0.000000
25%,28.000000,29900.000000,1.000000,4.770000e+04,0.000000
50%,33.000000,49900.000000,2.000000,7.990000e+04,0.000000
75%,38.250000,79900.000000,2.000000,1.299000e+05,0.000000
max,999.000000,129900.000000,200.000000,5.000000e+07,322778.000000


In [6]:
# 결측 진단
print("[열별 결측]")
display(missing_summary(partner))

# 결측이 채널과 관련 있는지 (MAR 신호 검사)
amt_null = partner[partner["amount"].isnull()]
print("\n[amount 결측 행의 채널 분포]")
print(amt_null["channel"].value_counts(normalize=True).round(2))
print("\n[전체 채널 분포]")
print(partner["channel"].value_counts(normalize=True).round(2))

[열별 결측]


,missing,missing_pct(%)
amount,51,3.40
price,5,0.33



[amount 결측 행의 채널 분포]
channel
app    0.88
web    0.12
Name: proportion, dtype: float64

[전체 채널 분포]
channel
app    0.61
web    0.39
Name: proportion, dtype: float64


In [7]:
# IQR 이상치 — 수치형 컬럼 일괄 점검
num_cols = ["customer_age", "price", "quantity", "amount", "return_amount"]
print("[IQR 기준 이상치 개수]")
for c in num_cols:
    series = partner[c].dropna()
    mask, (lo, up) = detect_outliers_iqr(series)
    print(f"  {c:15s}  하한={lo:>12.1f}  상한={up:>12.1f}  이상치={mask.sum()}건")
    if mask.sum() > 0:
        outlier_idx = series[mask].index
        display( partner.loc[outlier_idx] )
        
print("이상치 개수 체크")
display(partner[(partner["customer_age"] > 900) | (partner["customer_age"] < 10) ])
display(partner[(partner["quantity"] > 100) ])
display(partner[(partner["amount"] > 10_000_000) ])


[IQR 기준 이상치 개수]
  customer_age     하한=        12.6  상한=        53.6  이상치=18건


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
3,K00004,12,상의,web,49900.0,3,149700.0,0.0
75,K00076,999,상의,web,79900.0,1,79900.0,63174.0
91,K00092,999,액세서리,web,29900.0,3,89700.0,0.0
296,K00297,12,신발,app,29900.0,3,89700.0,0.0
417,K00418,5,상의,web,49900.0,3,149700.0,0.0
636,K00637,12,상의,web,79900.0,1,79900.0,0.0
703,K00704,60,신발,web,49900.0,1,49900.0,0.0
719,K00720,0,하의,web,129900.0,2,259800.0,0.0
757,K00758,54,액세서리,web,49900.0,3,149700.0,0.0
767,K00768,11,액세서리,web,15900.0,1,15900.0,0.0


  price            하한=    -45100.0  상한=    154900.0  이상치=0건
  quantity         하한=        -0.5  상한=         3.5  이상치=1건


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
24,K00025,38,신발,app,29900.0,200,89700.0,0.0


  amount           하한=    -75600.0  상한=    253200.0  이상치=145건


...

,order_id,customer_age,category,channel,price,quantity,amount,return_amount
0,K00001,47,신발,app,29900.0,2,59800.0,45445.0
38,K00039,21,신발,app,79900.0,2,159800.0,110296.0
64,K00065,50,하의,app,15900.0,1,15900.0,8746.0
65,K00066,26,하의,web,79900.0,2,159800.0,84493.0
75,K00076,999,상의,web,79900.0,1,79900.0,63174.0
...,...,...,...,...,...,...,...,...
1468,K01469,24,신발,web,15900.0,3,47700.0,45956.0
1477,K01478,32,상의,web,15900.0,2,31800.0,29538.0
1482,K01483,36,액세서리,web,29900.0,2,59800.0,46124.0
1483,K01484,34,상의,app,79900.0,2,159800.0,87832.0


이상치 개수 체크


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
75,K00076,999,상의,web,79900.0,1,79900.0,63174.0
91,K00092,999,액세서리,web,29900.0,3,89700.0,0.0
417,K00418,5,상의,web,49900.0,3,149700.0,0.0
719,K00720,0,하의,web,129900.0,2,259800.0,0.0
1250,K01251,6,상의,web,129900.0,3,389700.0,0.0
1264,K01265,999,상의,web,49900.0,2,99800.0,82390.0
1436,K01437,0,하의,web,15900.0,2,31800.0,0.0


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
24,K00025,38,신발,app,29900.0,200,89700.0,0.0


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
547,K00548,33,신발,app,49900.0,1,50000000.0,0.0


In [8]:
# 시나리오 3 — 처리 코드 (예시 구현)
partner_clean = partner.copy()

# 1) customer_age 물리적 불가능 값 → NaN → 중앙값 대체
unrealistic = (partner_clean["customer_age"] < 1) | (partner_clean["customer_age"] > 110)
print(f"#1 customer_age 수정 개수 : {unrealistic.sum()}")
partner_clean.loc[unrealistic, "customer_age"] = np.nan
partner_clean["customer_age"] = partner_clean["customer_age"].fillna(
    partner_clean["customer_age"].median()
).astype(int)

# 2) quantity 이상치 → NaN → 중앙값 대체
mask_q, _ = detect_outliers_iqr(partner_clean["quantity"])
print(f"#2 quantity 수정 개수 : {mask_q.sum()}")
partner_clean.loc[mask_q, "quantity"] = np.nan
partner_clean["quantity"] = partner_clean["quantity"].fillna(
    partner_clean["quantity"].median()
).astype(int)

# 3) amount 이상치(50,000,000) → 유지 + 플래그
mask_a, _ = detect_outliers_iqr(partner_clean["amount"])
partner_clean["amount_outlier"] = mask_a.astype(int)
print(f"#3 amount flag 개수 : {mask_a.sum()} {partner_clean['amount_outlier'].sum()}" )

# 4) amount 결측 → 채널별 중앙값 대체 (MAR 가설)
partner_clean["amount"] = partner_clean["amount"].fillna(
    partner_clean.groupby("channel")["amount"].transform("median")
)

# 5) price 결측 → 카테고리별 중앙값 대체
partner_clean["price"] = partner_clean["price"].fillna(
    partner_clean.groupby("category")["price"].transform("median")
)

# 검증 출력
print("[처리 전 후 결측 비교]")
before = partner.isnull().sum()
after = partner_clean[partner.columns].isnull().sum()
display(pd.DataFrame({"before": before, "after": after}))

print("\n[처리 후 customer_age 범위]:",
      partner_clean["customer_age"].min(), "~", partner_clean["customer_age"].max())
print("[amount_outlier=1 건수]:", partner_clean["amount_outlier"].sum())

#1 customer_age 수정 개수 : 5
#2 quantity 수정 개수 : 1
#3 amount flag 개수 : 145 145
[처리 전 후 결측 비교]


,before,after
order_id,0,0
customer_age,0,0
category,0,0
channel,0,0
price,5,0
quantity,0,0
amount,51,0
return_amount,0,0



[처리 후 customer_age 범위]: 5 ~ 60
[amount_outlier=1 건수]: 145


# 옷장패션 데이터 정제 보고서

## 1. 데이터 개요
- 행/열: 1,500 × 8
- 주요 컬럼: order_id, customer_age, category, channel, price, quantity, amount, return_amount

## 2. 진단 결과
- 결측: amount 3.4%, price 0.33%
- 이상치(IQR): customer_age(999·0, 5건), quantity(200, 1건), amount(50,000,000, 1건)
- 의심되는 결측 유형: amount는 app 채널에 쏠림 → MAR, price는 소량 결측, 무작위로 보임 -> MACR

## 3. 처리 결정과 근거
| 이슈 | 결정 | 근거 | 한계 |
- amount 결측 | 채널별 중앙값 대체 | app 채널의 결측 비율이 높음 -> MAR 가설 부합 | 채널 외 다른 원인은 검토 안 함 
- price 결측 | 카테고리별 중앙값 대체 | 비율이 낮고 카테고리별 가격대 보존 | 표본 적어 카테고리 통계 불안정 
- customer_age 이상치 (999,0) | NaN 표시 -> 중앙값 대체 | 물리적으로 불가능한 나이, 입력 실수 추정 | 외부 인증 데이터 없음 
- quantity 이상치 (200) | NaN 표시 -> 중앙값 대체 | 일반 소비자 외의 값으로 보임 | 도매 물량인지 확인이 불가능 (도매이면 제거 손실) 
- amount 이상치 (50,000,000) | 유지하고 flag에 표시 | 도매 물량일 가능성이 있음 | 도매 물량인지 세부 검토 안 함  

## 4. 처리 후 검증
- 결측 0건(설계상 NaN 유지가 필요한 컬럼 제외)
- customer_age 범위: 5 ~ 60 (정상)
- amount_outlier 플래그 145건 보존

## 5. 후속 권고
- outlier 추가 검증을 위해, customer_id 단위 과거 이력 확보 및 도매 가능 고객 식별 필요함 